# 01 — Data Health Monitor

**Purpose**: Monitor pipeline integrity and data collection health.

All data sourced from SQLite (`outputs/prediction_agent.db`) and JSONL files.

**Sections**:
1. Snapshot count over time
2. Missing snapshot detection
3. Snapshot hash uniqueness check
4. Market coverage breakdown
5. SQLite row count checks
6. Alert-style anomaly detection

In [ ]:
import sys
from pathlib import Path

# Add repo root to path
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone, timedelta

from config import OUTPUTS_DIR, SQLITE_DB_FILE
from prediction_agent.storage.sqlite_store import SQLiteStore

store = SQLiteStore()
print(f"DB path: {SQLITE_DB_FILE}")
print(f"DB exists: {SQLITE_DB_FILE.exists()}")

## 1. Snapshot Count Over Time

In [ ]:
snap_path = OUTPUTS_DIR / 'market_snapshots.jsonl'
snapshots = []
if snap_path.exists():
    with open(snap_path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                row = json.loads(line)
                snapshots.append(row)
            except: pass

print(f"Total snapshots loaded: {len(snapshots)}")

if snapshots:
    df = pd.DataFrame(snapshots)
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    df = df.dropna(subset=['timestamp'])
    df = df.sort_values('timestamp')

    # Resample to hourly counts
    df_hourly = df.set_index('timestamp').resample('1H').size().reset_index(name='count')

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(df_hourly['timestamp'], df_hourly['count'], alpha=0.4, color='steelblue')
    ax.plot(df_hourly['timestamp'], df_hourly['count'], color='steelblue', linewidth=1.5)
    ax.set_title('Market Snapshots Collected Per Hour', fontsize=13)
    ax.set_xlabel('Time (UTC)')
    ax.set_ylabel('Snapshot Count')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

    print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"Span: {(df['timestamp'].max() - df['timestamp'].min())}")
else:
    print('⚠ No snapshots found. Start the live loop to collect data.')

## 2. Missing Snapshot Detection

Checks for gaps larger than 2× the expected poll interval.

In [ ]:
from config import WATCHER_POLL_INTERVAL_SEC
EXPECTED_INTERVAL_MINUTES = WATCHER_POLL_INTERVAL_SEC / 60
GAP_THRESHOLD_MINUTES = EXPECTED_INTERVAL_MINUTES * 3  # flag gaps > 3× expected

if len(snapshots) > 1:
    timestamps = df[['timestamp']].drop_duplicates().sort_values('timestamp')
    timestamps['gap_minutes'] = timestamps['timestamp'].diff().dt.total_seconds() / 60
    gaps = timestamps[timestamps['gap_minutes'] > GAP_THRESHOLD_MINUTES]

    if len(gaps) == 0:
        print(f'✓ No gaps larger than {GAP_THRESHOLD_MINUTES:.0f}min detected.')
    else:
        print(f'⚠ {len(gaps)} gap(s) detected (> {GAP_THRESHOLD_MINUTES:.0f}min):')
        print(gaps[['timestamp', 'gap_minutes']].to_string(index=False))
        
    print(f'\nMedian gap: {timestamps["gap_minutes"].median():.1f} min')
    print(f'Max gap:    {timestamps["gap_minutes"].max():.1f} min')
else:
    print('Insufficient data for gap analysis.')

## 3. Snapshot Hash Uniqueness Check

In [ ]:
if snapshots:
    hashes = [s.get('_hash') for s in snapshots if s.get('_hash')]
    total = len(snapshots)
    hashed = len(hashes)
    unique = len(set(hashes))
    dups = hashed - unique

    print(f'Total snapshots:    {total}')
    print(f'With hash field:    {hashed}')
    print(f'Unique hashes:      {unique}')
    print(f'Duplicate hashes:   {dups}')

    if dups == 0:
        print('✓ All hashed snapshots are unique.')
    else:
        dup_pct = 100 * dups / hashed if hashed > 0 else 0
        print(f'⚠ {dups} duplicate snapshots ({dup_pct:.1f}%). Check deduplication logic.')
else:
    print('No snapshots to check.')

## 4. Market Coverage Breakdown

In [ ]:
if snapshots:
    market_counts = df['market_id'].value_counts().head(20)

    fig, ax = plt.subplots(figsize=(12, 5))
    market_counts.plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
    ax.set_title('Top 20 Markets by Snapshot Count', fontsize=13)
    ax.set_xlabel('Snapshot Count')
    ax.set_ylabel('Market ID')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    print(f'Unique markets tracked: {df["market_id"].nunique()}')
    print(f'Most sampled: {market_counts.index[0]} ({market_counts.iloc[0]} snapshots)')
else:
    print('No snapshot data.')

## 5. SQLite Row Count Checks

In [ ]:
if SQLITE_DB_FILE.exists():
    runs_count       = store.query('SELECT COUNT(*) AS cnt FROM runs')[0]['cnt']
    tool_out_count   = store.query('SELECT COUNT(*) AS cnt FROM tool_outputs')[0]['cnt']
    tools_count      = store.query('SELECT COUNT(*) AS cnt FROM tools')[0]['cnt']
    bets_triggered   = store.query('SELECT COUNT(*) AS cnt FROM runs WHERE bet_triggered=1')[0]['cnt']
    with_outcome     = store.query('SELECT COUNT(*) AS cnt FROM runs WHERE outcome IS NOT NULL')[0]['cnt']

    print('SQLite Table Row Counts')
    print('─' * 35)
    print(f'  runs:          {runs_count:>8,}')
    print(f'  tool_outputs:  {tool_out_count:>8,}')
    print(f'  tools:         {tools_count:>8,}')
    print(f'  bets_triggered:{bets_triggered:>8,}')
    print(f'  with_outcome:  {with_outcome:>8,}')

    # Consistency check: each run should have ≥1 tool_output
    r = store.query('SELECT COUNT(DISTINCT run_id) AS cnt FROM tool_outputs')
    runs_with_outputs = r[0]['cnt']
    if runs_with_outputs < runs_count:
        missing = runs_count - runs_with_outputs
        print(f'\n⚠ {missing} run(s) have no tool_outputs rows!')
    else:
        print('\n✓ All runs have corresponding tool_outputs.')
else:
    print('⚠ SQLite database not found. Run the live loop to generate data.')

## 6. Alert-Style Anomaly Detection

In [ ]:
alerts = []

# Alert 1: No data in last 2 hours
if SQLITE_DB_FILE.exists():
    r = store.query(
        "SELECT COUNT(*) AS cnt FROM runs WHERE timestamp >= datetime('now', '-2 hours')"
    )
    if r[0]['cnt'] == 0:
        alerts.append('🔴 No runs in last 2 hours — is the live loop running?')
    else:
        alerts.append(f'✓ {r[0]["cnt"]} run(s) in last 2 hours.')

# Alert 2: Bet trigger rate sanity check (should be between 1% and 90%)
if SQLITE_DB_FILE.exists():
    r_total = store.query('SELECT COUNT(*) AS cnt FROM runs')
    r_bets = store.query('SELECT COUNT(*) AS cnt FROM runs WHERE bet_triggered=1')
    total = r_total[0]['cnt']
    bets = r_bets[0]['cnt']
    if total > 0:
        rate = bets / total
        if rate > 0.90:
            alerts.append(f'🟡 High bet trigger rate: {rate:.1%}. Threshold may be too low.')
        elif rate < 0.01 and total > 20:
            alerts.append(f'🟡 Low bet trigger rate: {rate:.1%}. Threshold may be too high.')
        else:
            alerts.append(f'✓ Bet trigger rate: {rate:.1%} ({bets}/{total} runs).')

# Alert 3: Snapshot file size
snap_path = OUTPUTS_DIR / 'market_snapshots.jsonl'
if snap_path.exists():
    mb = snap_path.stat().st_size / (1024*1024)
    if mb > 500:
        alerts.append(f'🟡 Snapshot file is {mb:.0f}MB — consider archiving old data.')
    else:
        alerts.append(f'✓ Snapshot file size: {mb:.1f}MB.')
else:
    alerts.append('🔴 market_snapshots.jsonl not found.')

print('\n=== ALERTS ===')
for a in alerts:
    print(f'  {a}')